# GENIE systematic covariances (notebook production)

Compute per-knob **rate** and **cross-section** covariance packs for final-selection GENIE MC,
matching ``get_systematics_genie.py`` (legacy :func:`get_systematics` path) and
``syst_genie_aggregate.py`` disk layout.

- Loads ``evt`` + ``mcnu`` from ``dataset_locations.GENIE_GROUP_GLOBS`` (one knob group at a time).
- Writes per-group NPZs under ``systematics-genie-<date>/`` and ``GENIE/cov_mat_dict.pkl`` on the syst disk.
- Optional downstream cells load the pickle and reproduce Ar23 vs Ar23+ plots from the legacy notebook.

Set ``GENIE_GROUPS`` (env or config cell) to limit groups, e.g. ``CCQE,MEC``.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from os import path, makedirs
from datetime import datetime
from pathlib import Path
import json
import os
import gc
import pickle
import time
import warnings

import numpy as np
import pandas as pd
from pandas.errors import PerformanceWarning
from tqdm import tqdm

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')

from pyanalib.split_df_helpers_new import dfs_from_dir
from pyanalib.variable_calculator import (
    add_mc_cc1p0pi_tki_mcnu,
    add_reco_cc1p0pi_tki_evtdf,
    add_truth_cc1p0pi_tki_evtdf,
)

from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import plot_frac_unc, plot_heatmap
from analysis_village.numucc_1p0pi.files_config import save_fig_base_dir
from analysis_village.numucc_1p0pi.dataset_locations import (
    GENIE_GROUP_KNOBS,
    GENIE_GROUP_ORDER,
    _genie_glob_map,
)
from analysis_village.numucc_1p0pi.evt_derived_kinematics import ensure_derived_trk_kinematics_cols
from analysis_village.numucc_1p0pi.scripts.get_systematics_genie import (
    SystName,
    _align_evt_mcnu,
    _annotate_topo_genie_phi,
    genie_all_var_configs,
    genie_final_var_configs,
    get_systematics,
    sanitize_matrix_pack as _sanitize_matrix_pack,
)
from analysis_village.numucc_1p0pi.scripts.syst_genie_aggregate import (
    _init_cov_mat_dict,
    run_genie_syst_aggregate,
)
from analysis_village.numucc_1p0pi.syst_disk_layout import (
    FILE_GENIE,
    SUB_GENIE,
    category_out_dir,
    normalized_root,
)

warnings.filterwarnings('ignore', category=PerformanceWarning)
import matplotlib.pyplot as plt
plt.style.use('presentation.mplstyle')


def _ts():
    return datetime.now().strftime('%H:%M:%S')


def _log(msg):
    print(f'[{_ts()}] {msg}', flush=True)

In [ ]:
# --- run configuration (override via env) ---
INPUT_STAGE = os.environ.get('GENIE_MC_DF_STAGE', 'final')  # final | sel_all
_genie_groups_env = os.environ.get('GENIE_GROUPS', '')  # e.g. CCQE,MEC or empty = all with globs
XSEC_UNIT = float(os.environ.get('GENIE_XSEC_UNIT', '1.0'))
BKGD_SUBTRACT = os.environ.get('GENIE_BKGD_SUBTRACT', '1') not in ('0', 'false', 'False')
# MAX_FILES_PER_GROUP = int(os.environ.get('GENIE_MAX_FILES', '999'))
MAX_FILES_PER_GROUP = int(os.environ.get('GENIE_MAX_FILES', '25'))

AR23_GROUPS = frozenset({'CCQE', 'MEC', 'RES', 'nonRES', 'DIS', 'Other'})

gmap = _genie_glob_map(INPUT_STAGE)
if _genie_groups_env.strip():
    GENIE_GROUPS = tuple(x.strip() for x in _genie_groups_env.split(',') if x.strip())
else:
    GENIE_GROUPS = tuple(g for g in GENIE_GROUP_ORDER if g in gmap)

GENIE_GROUPS = ('CCQE', 'MEC', 'RES', 'nonRES', 'DIS', 'Other', 'Ar23p')

bad = [g for g in GENIE_GROUPS if g not in gmap]
if bad:
    raise KeyError(f'unknown or empty GENIE group(s) {bad}; globs for stage={INPUT_STAGE!r}: {tuple(gmap)}')

var_configs = genie_all_var_configs(INPUT_STAGE) if INPUT_STAGE != 'final' else genie_final_var_configs()
_log(f'input_stage={INPUT_STAGE} groups={GENIE_GROUPS}')
_log(f'{len(var_configs)} variables; xsec_unit={XSEC_UNIT} bkgd_subtract={BKGD_SUBTRACT}')

In [ ]:
def genie_merged_search_dir(group: str) -> str:
    # parent of merged_perTPC/*.df
    return str(Path(gmap[group]).parent)


def load_genie_group_frames(group: str):
    search_dir = genie_merged_search_dir(group)
    _log(f'loading {group} from {search_dir} ...')
    t0 = time.time()
    dfs = dfs_from_dir(
        search_dir=search_dir,
        filename_str='sel_mup',
        keys2load=['evt', 'mcnu'],
        n_max_concat=MAX_FILES_PER_GROUP,
        # load all HDF splits per file (n_max_concat limits *files* only)
        n_max_splits_per_file=None,
    )
    if 'evt' not in dfs or 'mcnu' not in dfs:
        raise RuntimeError(f'missing evt/mcnu for {group} under {search_dir}')
    evt, mcnu = _align_evt_mcnu(dfs['evt'], dfs['mcnu'])
    _log(f'  {group}: {len(evt):,} aligned evt rows in {time.time() - t0:.1f}s')
    return evt, mcnu


def prepare_genie_frames(evt: pd.DataFrame, mcnu: pd.DataFrame):
    # Same derived columns / phi fill as get_systematics_genie.run_chunk_map (final).
    evt = evt.copy()
    mcnu = mcnu.copy()
    evt = ensure_derived_trk_kinematics_cols(evt)
    evt = add_reco_cc1p0pi_tki_evtdf(evt)
    evt = add_truth_cc1p0pi_tki_evtdf(evt)
    mcnu = add_mc_cc1p0pi_tki_mcnu(mcnu)
    _annotate_topo_genie_phi(evt, mcnu)
    return _align_evt_mcnu(evt, mcnu)


def frac_unc_from_pack(pack):
    return np.sqrt(np.maximum(np.diag(pack['cov_frac']), 0.0))

In [ ]:
today_str = datetime.now().strftime('%Y%m%d')
today_str = "integrated"
NOTEBOOK_OUT_ROOT = path.join(save_fig_base_dir, f'systematics-genie-{today_str}')
makedirs(NOTEBOOK_OUT_ROOT, exist_ok=True)

SYST_DISK_ROOT = path.join(save_fig_base_dir, f'systematics-notebook-genie-{today_str}')
genie_disk_dir = category_out_dir(SYST_DISK_ROOT, SUB_GENIE)
makedirs(genie_disk_dir, exist_ok=True)
COV_MAT_PKL = path.join(genie_disk_dir, FILE_GENIE)

_log('notebook NPZ root: ' + normalized_root(NOTEBOOK_OUT_ROOT))
_log('syst disk GENIE dir: ' + normalized_root(genie_disk_dir))

PLOT_HEATMAP_FIRST_KNOB = False
SAVE_PER_GROUP_NPZ = True

In [ ]:
def syst_dict_var_first(group_syst_by_var):
    return {slug: np.array(by_knob, dtype=object) for slug, by_knob in group_syst_by_var.items()}


def syst_dict_knob_first(group_syst_by_knob):
    return {knob: np.array(by_var, dtype=object) for knob, by_var in group_syst_by_knob.items()}


def run_group_in_memory(group: str, cov_mat_dict):
    knobs = list(GENIE_GROUP_KNOBS.get(group, ()))
    if not knobs:
        _log(f'SKIP {group}: no knobs in GENIE_GROUP_KNOBS')
        return None

    evt, mcnu = load_genie_group_frames(group)
    mc_evt, mc_nu = prepare_genie_frames(evt, mcnu)
    del evt, mcnu
    gc.collect()

    group_syst_by_var = {vc.var_save_name: {} for vc in var_configs}
    group_syst_by_knob = {k: {} for k in knobs}
    t_group = time.time()

    for iknob, knob in enumerate(tqdm(knobs, desc=f'knobs ({group})')):
        syst_name: SystName = ('mc', knob)
        for ivar, vc in enumerate(var_configs):
            slug = vc.var_save_name
            if iknob == 0:
                _log(f'--- {group} variable [{ivar + 1}/{len(var_configs)}] {slug} ---')
            try:
                matrices = get_systematics(
                    mc_evt,
                    mc_nu,
                    vc,
                    syst_name,
                    syst_type='GENIE',
                    plot=False,
                    xsec_unit=XSEC_UNIT,
                )
            except Exception as ex:
                _log(f'  SKIP {group}/{knob}/{slug}: {ex}')
                continue
            rate_pack = _sanitize_matrix_pack(matrices['rate'])
            xsec_pack = _sanitize_matrix_pack(matrices['xsec'])
            pack = {'rate': rate_pack, 'xsec': xsec_pack}
            group_syst_by_var[slug][knob] = pack
            group_syst_by_knob[knob][slug] = pack

            row = cov_mat_dict.get(slug)
            if row is None:
                continue
            cf_r = pack['rate']['cov_frac']
            rk = f'{knob}_rate'
            if rk not in row:
                row[rk] = np.zeros_like(cf_r)
            row[rk] += cf_r
            row['genie_rate'] += cf_r
            if group in AR23_GROUPS:
                row['genie_ar23_rate'] += cf_r
            cf_x = pack['xsec']['cov_frac']
            if knob not in row:
                row[knob] = np.zeros_like(cf_x)
            row[knob] += cf_x
            row['genie'] += cf_x
            if group in AR23_GROUPS:
                row['genie_ar23'] += cf_x

            if PLOT_HEATMAP_FIRST_KNOB and iknob == 0 and ivar == 0:
                plot_heatmap(
                    xsec_pack['cov_frac'],
                    vc.bins,
                    plot_labels=[vc.var_labels[1], vc.var_labels[1], 'cov_frac (xsec)'],
                    plot=True,
                )
                plt.suptitle(f'{group} / {knob} / {slug}')
                plt.tight_layout()
                plt.show()

    group_syst_by_var = {s: d for s, d in group_syst_by_var.items() if d}
    group_syst_by_knob = {k: d for k, d in group_syst_by_knob.items() if d}
    _log(f'FINISHED {group} in {time.time() - t_group:.1f}s')

    if SAVE_PER_GROUP_NPZ and group_syst_by_var:
        grp_dir = path.join(NOTEBOOK_OUT_ROOT, f'systematics-genie-{group}-{today_str}')
        makedirs(grp_dir, exist_ok=True)
        if group == 'Ar23p':
            npz_path = path.join(grp_dir, 'genie-Ar23p_syst_dict.npz')
            np.savez_compressed(npz_path, **syst_dict_knob_first(group_syst_by_knob))
        else:
            npz_path = path.join(grp_dir, f'genie-{group}_syst_dict.npz')
            np.savez_compressed(npz_path, **syst_dict_var_first(group_syst_by_var))
        _log(f'  wrote {npz_path}')

    del mc_evt, mc_nu
    gc.collect()
    return group_syst_by_var

In [ ]:
# GENIE_GROUPS = ('Ar23p',)
var_configs = [VariableConfig.all_events(),
                # VariableConfig.muon_momentum(),
                # VariableConfig.muon_direction(),
                # VariableConfig.proton_momentum(),
                # VariableConfig.proton_direction(),
                # VariableConfig.tki_del_alpha(),
                # VariableConfig.tki_del_phi(),
                # VariableConfig.tki_del_Tp(),
                # VariableConfig.tki_del_p(),
                # VariableConfig.tki_del_Tp_x(),
                # VariableConfig.tki_del_Tp_y(),
                # VariableConfig.muon_direction_x(),
                # VariableConfig.muon_direction_y(),
                # VariableConfig.proton_direction_x(),
                # VariableConfig.proton_direction_y(),
                # # VariableConfig.opening_angle(),
                # VariableConfig.vertex_x(),
                # VariableConfig.vertex_y(),
                # VariableConfig.vertex_z(),
                ]

In [ ]:
cov_mat_dict = _init_cov_mat_dict(var_configs)
for slug in cov_mat_dict:
    n = len(next(vc for vc in var_configs if vc.var_save_name == slug).bin_centers)
    z = np.zeros((n, n), dtype=np.float64)
    cov_mat_dict[slug]['genie_ar23'] = z.copy()
    cov_mat_dict[slug]['genie_ar23_rate'] = z.copy()

t_all = time.time()
for group in GENIE_GROUPS:
    _log(f'=== GENIE group {group} ===')
    run_group_in_memory(group, cov_mat_dict)

with open(COV_MAT_PKL, 'wb') as f:
    pickle.dump(cov_mat_dict, f, protocol=pickle.HIGHEST_PROTOCOL)
_log(f'wrote {COV_MAT_PKL} in {time.time() - t_all:.1f}s')

manifest = {
    'schema': 'numucc_genie_cov_mat_dict_v1',
    'description': 'Notebook GENIE production via get_systematics_genie.get_systematics',
    'mc_df_stage': INPUT_STAGE,
    'genie_groups': list(GENIE_GROUPS),
    'variables': sorted(cov_mat_dict.keys()),
    'xsec_unit': XSEC_UNIT,
    'bkgd_subtract': BKGD_SUBTRACT,
    'output_pkl': COV_MAT_PKL,
    'per_group_npz_root': NOTEBOOK_OUT_ROOT,
}
manifest_path = path.join(genie_disk_dir, 'genie_covariance_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
_log('wrote ' + manifest_path)

## Integrated xsec decomposition (efficiency / smearing / background)

Cross-section universes use ``R(reco, eff) @ N_gen^CV`` plus background subtraction, with **fixed**
``N_gen^CV``. For the single-bin ``integrated`` variable, smearing is identically 1×1 (no migration),
so the **smearing** component should be ~0 by construction.

Set ``RUN_XSEC_DECOMP`` and optionally ``DECOMP_GROUPS`` / ``DECOMP_MAX_KNOBS_PER_GROUP``.


In [ ]:
# --- integrated xsec diagnostics: efficiency vs smearing vs background ---
from analysis_village.numucc_1p0pi.scripts.get_systematics_genie import (
    XSEC_COMPONENTS,
    _empty_xsec_tensor_acc,
    accumulate_xsec_path_chunk,
    combine_component_cov_fracs,
    normalize_and_infer_n_univ,
    print_xsec_component_table,
    xsec_path_component_diagnostics,
)

RUN_XSEC_DECOMP = True
# DECOMP_GROUPS = ('CCQE',)  # start with one group; use GENIE_GROUPS for all
DECOMP_GROUPS = GENIE_GROUPS  # start with one group; use GENIE_GROUPS for all
DECOMP_MAX_KNOBS_PER_GROUP = None  # None = all knobs in group
DECOMP_OUT_DIR = path.join(NOTEBOOK_OUT_ROOT, 'xsec_decomp_integrated')
makedirs(DECOMP_OUT_DIR, exist_ok=True)

vc_int = VariableConfig.all_events()

if RUN_XSEC_DECOMP:
    all_diags = []
    for group in DECOMP_GROUPS:
        knobs = list(GENIE_GROUP_KNOBS.get(group, ()))
        if DECOMP_MAX_KNOBS_PER_GROUP is not None:
            knobs = knobs[: int(DECOMP_MAX_KNOBS_PER_GROUP)]
        _log(f'xsec decomp: loading {group} ({len(knobs)} knobs) ...')
        evt, mcnu = load_genie_group_frames(group)
        mc_evt, mc_nu = prepare_genie_frames(evt, mcnu)
        del evt, mcnu
        gc.collect()
        group_diags = []
        for knob in tqdm(knobs, desc=f'decomp {group}'):
            syst_name = ('mc', knob)
            n_univ = normalize_and_infer_n_univ(mc_evt, mc_nu, syst_name)
            acc = _empty_xsec_tensor_acc(n_univ, len(vc_int.bin_centers))
            accumulate_xsec_path_chunk(mc_evt, mc_nu, vc_int, syst_name, n_univ, acc)
            diag = xsec_path_component_diagnostics(
                acc, vc_int, syst_name, xsec_unit=XSEC_UNIT, bkgd_subtract=BKGD_SUBTRACT,
            )
            group_diags.append(diag)
        all_diags.extend(group_diags)
        print_xsec_component_table(group_diags, title=f'=== {group} per-knob xsec unc [%] ===')

    print_xsec_component_table(all_diags, title='=== Combined groups (indep sum) ===')

    # Compare to cov_mat_dict totals if available
    if 'cov_mat_dict' in dir() and 'integrated' in cov_mat_dict:
        row = cov_mat_dict['integrated']
        g = row.get('genie')
        gr = row.get('genie_rate')
        if g is not None:
            print(f"pickle genie (xsec total):     {100*np.sqrt(max(float(g.flat[0]),0)):.4f}%")
        if gr is not None:
            print(f"pickle genie_rate:           {100*np.sqrt(max(float(gr.flat[0]),0)):.4f}%")
    if path.isfile(COV_MAT_PKL):
        with open(COV_MAT_PKL, 'rb') as f:
            _pkl = pickle.load(f)
        if 'integrated' in _pkl:
            print(f"disk pickle genie (xsec):      {100*np.sqrt(max(float(_pkl['integrated']['genie'].flat[0]),0)):.4f}%")
            print(f"disk pickle genie_rate:        {100*np.sqrt(max(float(_pkl['integrated']['genie_rate'].flat[0]),0)):.4f}%")

    # --- plots ---
    comps_plot = ['full', 'efficiency', 'smearing', 'background', 'signal']
    summed = {c: 100*np.sqrt(max(float(combine_component_cov_fracs(all_diags, c).flat[0]), 0)) for c in comps_plot}

    fig, ax = plt.subplots(figsize=(8, 4))
    xs = np.arange(len(comps_plot))
    ax.bar(xs, [summed[c] for c in comps_plot], color=['k', 'C0', 'C1', 'C2', 'C3'])
    ax.set_xticks(xs, [c.replace('efficiency', 'eff').replace('background', 'bkg') for c in comps_plot], rotation=15)
    ax.set_ylabel('integrated fractional unc [%]')
    ax.set_title('GENIE integrated xsec — component breakdown (indep knob sum)')
    for i, c in enumerate(comps_plot):
        ax.text(i, summed[c] + 0.05, f'{summed[c]:.2f}%', ha='center', va='bottom', fontsize=9)
    fig.tight_layout()
    out_bar = path.join(DECOMP_OUT_DIR, 'integrated_xsec_components_bar.png')
    fig.savefig(out_bar, dpi=150, bbox_inches='tight')
    plt.show()
    _log('saved ' + out_bar)

    # Example knob: universe pulls vs CV
    ex = max(all_diags, key=lambda d: d['full']['unc_pct'])
    cv0 = float(ex['cv_xsec'][0])
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
    axes = axes.flat
    for ax, comp in zip(axes, comps_plot):
        pulls = (ex[comp]['univ_events'][:, 0] - cv0) / cv0 if cv0 else ex[comp]['univ_events'][:, 0] * 0
        ax.hist(100 * pulls, bins=30, histtype='step', color='k')
        ax.set_title(f"{comp}\nσ_pull={ex[comp]['rel_pull_rms']*100:.3f}%  unc={ex[comp]['unc_pct']:.3f}%")
        ax.set_xlabel('(univ − CV) / CV [%]')
        ax.axvline(0, color='gray', ls='--', lw=0.8)
    fig.suptitle(f"Largest full-xsec knob: {ex['knob'][:60]}", y=1.02)
    fig.tight_layout()
    out_hist = path.join(DECOMP_OUT_DIR, 'integrated_xsec_pulls_example_knob.png')
    fig.savefig(out_hist, dpi=150, bbox_inches='tight')
    plt.show()
    _log('saved ' + out_hist)

    # Smearing check: max |reco_u - reco_cv| for integrated (should be 0)
    smear_max = 0.0
    for d in all_diags[:5]:
        smear_max = max(smear_max, abs(d['smearing']['rel_pull_max']))
    print(f"integrated smearing: max |rel pull| over first 5 knobs = {smear_max:.3e} (expect ~0)")
else:
    _log('RUN_XSEC_DECOMP=False (skipped)')


## Merge integrated GENIE into `systematics-final`

After re-running the production cell above with the fixed xsec path, replace only the
``integrated`` slug in the final pickle (rate and xsec are **not** the same matrix).

```bash
python analysis_village/numucc_1p0pi/scripts/merge_integrated_genie_into_final.py
```

Expect ``genie`` (xsec total) $\ll$ ``genie_rate`` (typically a few % vs $\sim$30%).


In [ ]:
# Or run in-notebook (same as merge_integrated_genie_into_final.py):
from analysis_village.numucc_1p0pi.scripts.merge_integrated_genie_into_final import merge_integrated_row
import pickle, shutil
from analysis_village.numucc_1p0pi.files_config import save_fig_base_dir
from analysis_village.numucc_1p0pi.syst_disk_layout import FILE_GENIE, SUB_GENIE, category_out_dir

FINAL_GENIE_PKL = path.join(category_out_dir(path.join(save_fig_base_dir, 'systematics-final'), SUB_GENIE), FILE_GENIE)
INTEGRATED_SRC = COV_MAT_PKL  # systematics-notebook-genie-integrated/GENIE/cov_mat_dict.pkl
BACKUP = FINAL_GENIE_PKL + '.pre_integrated_merge'

MERGE_INTEGRATED_INTO_FINAL = False  # set True after re-running production with fixed xsec
if MERGE_INTEGRATED_INTO_FINAL:
    if not path.isfile(BACKUP):
        shutil.copy2(FINAL_GENIE_PKL, BACKUP)
        _log('backup -> ' + BACKUP)
    with open(INTEGRATED_SRC, 'rb') as f:
        src_cov = pickle.load(f)
    with open(FINAL_GENIE_PKL, 'rb') as f:
        final_cov = pickle.load(f)
    merge_integrated_row(final_cov, src_cov)
    with open(FINAL_GENIE_PKL, 'wb') as f:
        pickle.dump(final_cov, f, protocol=pickle.HIGHEST_PROTOCOL)
    _log('merged integrated into ' + FINAL_GENIE_PKL)


## Optional: merge precomputed chunk pickles

If you already ran ``get_systematics_genie.py chunk-map`` on the grid, set ``CHUNKS_DIR`` and run the
next cell. This calls ``syst_genie_aggregate.run_genie_syst_aggregate`` (same as the batch merge step).

In [ ]:
# CHUNKS_DIR = '/path/to/genie_syst-chunked-YYYYMMDD/chunks'
# if CHUNKS_DIR:
#     run_genie_syst_aggregate(
#         CHUNKS_DIR,
#         SYST_DISK_ROOT,
#         mc_df_stage=INPUT_STAGE,
#         xsec_unit=XSEC_UNIT,
#     )

## Optional: build `cov_mat_dict` from existing per-group NPZs

Use this instead of the in-memory production cell when NPZs already exist (same layout as
`systematics-genie-old.ipynb`: var-first for Ar23 groups, knob-first for Ar23p).

In [ ]:
# LOAD_FROM_NPZ = True
# LOAD_DATE = '20260216'  # or today_str
# load_root = path.join(save_fig_base_dir, f'systematics-genie-{LOAD_DATE}')
#
# cov_mat_dict = _init_cov_mat_dict(var_configs)
# for slug in cov_mat_dict:
#     n = len(next(vc for vc in var_configs if vc.var_save_name == slug).bin_centers)
#     z = np.zeros((n, n), dtype=np.float64)
#     cov_mat_dict[slug]['genie_ar23'] = z.copy()
#     cov_mat_dict[slug]['genie_ar23_rate'] = z.copy()
#
# for genie_tag in [g for g in GENIE_GROUPS if g in AR23_GROUPS]:
#     unc_file = path.join(load_root, f'systematics-genie-{genie_tag}-{LOAD_DATE}', f'genie-{genie_tag}_syst_dict.npz')
#     unc = np.load(unc_file, allow_pickle=True)
#     for vc in var_configs:
#         slug = vc.var_save_name
#         by_knob = unc[slug].item()
#         for k, packs in by_knob.items():
#             cov_mat_dict[slug][k] = packs['xsec']['cov_frac']
#             cov_mat_dict[slug][k + '_rate'] = packs['rate']['cov_frac']
#             cov_mat_dict[slug]['genie'] += packs['xsec']['cov_frac']
#             cov_mat_dict[slug]['genie_rate'] += packs['rate']['cov_frac']
#             cov_mat_dict[slug]['genie_ar23'] += packs['xsec']['cov_frac']
#             cov_mat_dict[slug]['genie_ar23_rate'] += packs['rate']['cov_frac']
#
# if 'Ar23p' in GENIE_GROUPS:
#     unc_file = path.join(load_root, f'systematics-genie-Ar23p-{LOAD_DATE}', 'genie-Ar23p_syst_dict.npz')
#     unc = np.load(unc_file, allow_pickle=True)
#     for k in unc.files:
#         by_var = unc[k].item()
#         for slug, packs in by_var.items():
#             cov_mat_dict[slug][k] = packs['xsec']['cov_frac']
#             cov_mat_dict[slug][k + '_rate'] = packs['rate']['cov_frac']
#             cov_mat_dict[slug]['genie'] += packs['xsec']['cov_frac']
#             cov_mat_dict[slug]['genie_rate'] += packs['rate']['cov_frac']

## Inspect totals (Ar23 vs Ar23+)

Legacy comparison plots from ``systematics-genie-old.ipynb``.

In [ ]:
save_fig = False
save_fig_dir = path.join(save_fig_base_dir, f'systematics_studies_genie-{today_str}')
if save_fig:
    makedirs(save_fig_dir, exist_ok=True)

inspect_var = VariableConfig.tki_del_Tp()
slug = inspect_var.var_save_name
row = cov_mat_dict[slug]

ar23_unc = np.sqrt(np.maximum(np.diag(row['genie_ar23']), 0.0))
ar23_unc_rate = np.sqrt(np.maximum(np.diag(row['genie_ar23_rate']), 0.0))
ar23p_unc = np.sqrt(np.maximum(np.diag(row['genie']), 0.0))
ar23p_unc_rate = np.sqrt(np.maximum(np.diag(row['genie_rate']), 0.0))

print('Ar23 total rate frac unc (mean):', np.mean(ar23_unc_rate))
print('Ar23+ total rate frac unc (mean):', np.mean(ar23p_unc_rate))
print('Ar23 total xsec frac unc (mean):', np.mean(ar23_unc))
print('Ar23+ total xsec frac unc (mean):', np.mean(ar23p_unc))

plot_frac_unc(
    [ar23_unc_rate, ar23p_unc_rate],
    inspect_var,
    legends=['Ar23', 'Ar23+'],
    plot_labels=['', '', 'Total uncertainty on signal rate'],
)
plt.suptitle(f'{slug}: Ar23 vs Ar23+ rate', y=1.02)
plt.tight_layout()
plt.show()

plot_frac_unc(
    [ar23_unc, ar23p_unc],
    inspect_var,
    legends=['Ar23', 'Ar23+'],
    plot_labels=['', '', 'Total uncertainty on signal xsec'],
)
plt.suptitle(f'{slug}: Ar23 vs Ar23+ xsec', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: reload per-group NPZ (var-first layout) for one Ar23 group
# RELOAD_GROUP = 'CCQE'
# unc_path = path.join(
#     NOTEBOOK_OUT_ROOT,
#     f'systematics-genie-{RELOAD_GROUP}-{today_str}',
#     f'genie-{RELOAD_GROUP}_syst_dict.npz',
# )
# unc = np.load(unc_path, allow_pickle=True)
# slug = VariableConfig.muon_momentum().var_save_name
# knob = list(unc[slug].item().keys())[0]
# pack = unc[slug].item()[knob]
# print('sqrt(diag(cov_frac)) rate:', frac_unc_from_pack(pack['rate']))